# Liveliness Signal Backtest with VectorBT

A complete backtest and walk-forward validation of the liveliness signal.

## Contents
1. Setup and Data Loading
2. Signal Generation
3. Basic Backtest
4. Parameter Optimization (Grid Search)
5. Walk-Forward Validation
6. Performance Analysis
7. Comparison to Buy & Hold

---
## 1. Setup and Data Loading

In [ ]:
# Install vectorbt if needed
# !pip install vectorbt

In [ ]:
import pandas as pd
import numpy as np
import vectorbt as vbt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set vectorbt settings
vbt.settings.array_wrapper['freq'] = 'D'  # Daily frequency
vbt.settings.portfolio['init_cash'] = 100_000  # Starting capital

print(f"VectorBT version: {vbt.__version__}")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

# Load liveliness and price
liveliness = pd.read_parquet(DATA_DIR / "liveliness.parquet")
price = pd.read_parquet(DATA_DIR / "price.parquet")

# Prepare dataframe
liveliness = liveliness.rename(columns={"value": "liveliness"}).set_index("time")
price = price.rename(columns={"value": "price"}).set_index("time")

df = liveliness.join(price, how='inner').sort_index()

# Convert to Series for vectorbt
close = df['price']
liveliness_series = df['liveliness']

print(f"Data loaded: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"\nPrice range: ${close.min():,.0f} - ${close.max():,.0f}")
print(f"Liveliness range: {liveliness_series.min():.4f} - {liveliness_series.max():.4f}")

In [ ]:
# Quick visualization
fig = vbt.make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        subplot_titles=['Bitcoin Price', 'Liveliness'])

fig.add_trace(close.vbt.plot().data[0], row=1, col=1)
fig.add_trace(liveliness_series.vbt.plot().data[0], row=2, col=1)

fig.update_layout(height=600, showlegend=False)
fig.show()

---
## 2. Signal Generation

Based on regression analysis:
- **Negative coefficient** in 4/5 bull cycles
- **Signal:** BUY when liveliness < threshold (low liveliness = HODLers accumulating)

In [ ]:
# Define signal parameters from regression analysis
THRESHOLD = 0.49  # From grid search - buy when liveliness < this
DIRECTION = 'below'  # Negative coefficient means buy when BELOW

# Generate entry signals
# Entry: liveliness crosses below threshold (entering favorable zone)
# Exit: liveliness crosses above threshold (leaving favorable zone)

in_signal = liveliness_series < THRESHOLD

# Entry = transition from False to True (crossing below)
entries = in_signal & ~in_signal.shift(1).fillna(False)

# Exit = transition from True to False (crossing above)
exits = ~in_signal & in_signal.shift(1).fillna(False)

print(f"Threshold: {THRESHOLD}")
print(f"Direction: Buy when liveliness {DIRECTION} threshold")
print(f"\nTotal entry signals: {entries.sum()}")
print(f"Total exit signals: {exits.sum()}")
print(f"Days in signal (liveliness < {THRESHOLD}): {in_signal.sum()} ({in_signal.mean()*100:.1f}%)")

In [ ]:
# Visualize signals on price chart
fig = vbt.make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=['Price with Signals', 'Liveliness with Threshold'])

# Price
fig.add_scatter(x=close.index, y=close, name='Price', row=1, col=1)

# Entry markers
entry_prices = close[entries]
fig.add_scatter(x=entry_prices.index, y=entry_prices, mode='markers',
                marker=dict(symbol='triangle-up', size=10, color='green'),
                name='Entry', row=1, col=1)

# Exit markers
exit_prices = close[exits]
fig.add_scatter(x=exit_prices.index, y=exit_prices, mode='markers',
                marker=dict(symbol='triangle-down', size=10, color='red'),
                name='Exit', row=1, col=1)

# Liveliness
fig.add_scatter(x=liveliness_series.index, y=liveliness_series, 
                name='Liveliness', row=2, col=1)
fig.add_hline(y=THRESHOLD, line_dash='dash', line_color='red', row=2, col=1)

fig.update_layout(height=700)
fig.show()

---
## 3. Basic Backtest

In [ ]:
# Run backtest with vectorbt
pf = vbt.Portfolio.from_signals(
    close=close,
    entries=entries,
    exits=exits,
    init_cash=100_000,
    fees=0.001,  # 0.1% trading fee
    slippage=0.001,  # 0.1% slippage
    freq='D'
)

# Print stats
print(pf.stats())

In [ ]:
# Plot portfolio performance
pf.plot().show()

In [ ]:
# Compare to Buy & Hold
pf_hold = vbt.Portfolio.from_holding(close, init_cash=100_000, freq='D')

print("\n" + "="*60)
print("STRATEGY vs BUY & HOLD")
print("="*60)
print(f"{'Metric':<25} {'Strategy':>15} {'Buy & Hold':>15}")
print("-"*60)
print(f"{'Total Return':<25} {pf.total_return()*100:>14.1f}% {pf_hold.total_return()*100:>14.1f}%")
print(f"{'Sharpe Ratio':<25} {pf.sharpe_ratio():>15.2f} {pf_hold.sharpe_ratio():>15.2f}")
print(f"{'Sortino Ratio':<25} {pf.sortino_ratio():>15.2f} {pf_hold.sortino_ratio():>15.2f}")
print(f"{'Max Drawdown':<25} {pf.max_drawdown()*100:>14.1f}% {pf_hold.max_drawdown()*100:>14.1f}%")
print(f"{'Win Rate':<25} {pf.trades.win_rate()*100:>14.1f}% {'N/A':>15}")
print(f"{'Total Trades':<25} {pf.trades.count():>15} {'1':>15}")

In [ ]:
# Plot cumulative returns comparison
fig = vbt.make_subplots(rows=1, cols=1)

strat_returns = pf.cumulative_returns() * 100
hold_returns = pf_hold.cumulative_returns() * 100

fig.add_scatter(x=strat_returns.index, y=strat_returns, name='Liveliness Strategy')
fig.add_scatter(x=hold_returns.index, y=hold_returns, name='Buy & Hold')

fig.update_layout(
    title='Cumulative Returns: Strategy vs Buy & Hold',
    yaxis_title='Cumulative Return (%)',
    height=500
)
fig.show()

---
## 4. Parameter Optimization (Grid Search)

Test multiple thresholds to find the optimal one and check for smoothness.

In [ ]:
# Define threshold range to test
thresholds = np.linspace(
    liveliness_series.quantile(0.05),
    liveliness_series.quantile(0.95),
    20
)

print(f"Testing {len(thresholds)} thresholds")
print(f"Range: {thresholds.min():.4f} to {thresholds.max():.4f}")

In [ ]:
# Generate signals for all thresholds at once (vectorized!)
# This is the power of vectorbt - test many parameters simultaneously

# Create a DataFrame with signals for each threshold
entries_matrix = pd.DataFrame(index=close.index)
exits_matrix = pd.DataFrame(index=close.index)

for thresh in thresholds:
    in_sig = liveliness_series < thresh
    entries_matrix[thresh] = in_sig & ~in_sig.shift(1).fillna(False)
    exits_matrix[thresh] = ~in_sig & in_sig.shift(1).fillna(False)

print(f"Generated signals for {len(thresholds)} thresholds")
print(f"Entries shape: {entries_matrix.shape}")

In [ ]:
# Run backtest for all thresholds at once
pf_opt = vbt.Portfolio.from_signals(
    close=close,
    entries=entries_matrix,
    exits=exits_matrix,
    init_cash=100_000,
    fees=0.001,
    slippage=0.001,
    freq='D'
)

# Get performance metrics for each threshold
results = pd.DataFrame({
    'threshold': thresholds,
    'total_return': pf_opt.total_return().values,
    'sharpe': pf_opt.sharpe_ratio().values,
    'sortino': pf_opt.sortino_ratio().values,
    'max_dd': pf_opt.max_drawdown().values,
    'win_rate': pf_opt.trades.win_rate().values,
    'n_trades': pf_opt.trades.count().values
})

results

In [ ]:
# Plot Sharpe curve - CHECK FOR SMOOTHNESS
fig = vbt.make_subplots(rows=2, cols=2, 
                        subplot_titles=['Sharpe Ratio (should be smooth!)', 
                                       'Total Return', 
                                       'Max Drawdown',
                                       'Number of Trades'])

# Sharpe
fig.add_scatter(x=results['threshold'], y=results['sharpe'], 
                mode='lines+markers', name='Sharpe', row=1, col=1)
fig.add_hline(y=0, line_dash='dash', row=1, col=1)

# Total Return
fig.add_scatter(x=results['threshold'], y=results['total_return']*100, 
                mode='lines+markers', name='Return %', row=1, col=2)

# Max Drawdown
fig.add_scatter(x=results['threshold'], y=results['max_dd']*100, 
                mode='lines+markers', name='Max DD %', row=2, col=1)

# N Trades
fig.add_bar(x=results['threshold'], y=results['n_trades'], 
            name='Trades', row=2, col=2)

fig.update_layout(height=700, showlegend=False)
fig.show()

# Calculate smoothness score
sharpe_diffs = np.diff(results['sharpe'].fillna(0))
smoothness = np.std(sharpe_diffs) / (np.mean(np.abs(results['sharpe'].fillna(0))) + 1e-6)

print(f"\nSmoothness score: {smoothness:.2f}")
print(f"(Lower is better. < 0.5 is acceptably smooth)")
print(f"→ {'✓ SMOOTH - signal is robust' if smoothness < 0.5 else '✗ SPIKY - possible overfit'}")

In [ ]:
# Find optimal threshold
best_idx = results['sharpe'].idxmax()
best = results.loc[best_idx]

print("\n" + "="*60)
print("OPTIMAL THRESHOLD")
print("="*60)
print(f"Threshold: {best['threshold']:.4f}")
print(f"Sharpe Ratio: {best['sharpe']:.2f}")
print(f"Total Return: {best['total_return']*100:.1f}%")
print(f"Max Drawdown: {best['max_dd']*100:.1f}%")
print(f"Win Rate: {best['win_rate']*100:.1f}%")
print(f"Number of Trades: {best['n_trades']:.0f}")

---
## 5. Walk-Forward Validation

The TRUE test: optimize on past data, test on future data.

In [ ]:
# Define walk-forward parameters
TRAIN_PERIOD = 365  # 1 year training
TEST_PERIOD = 90    # 3 months testing
STEP = 90           # Roll forward every 3 months

# Calculate number of folds
total_days = len(close)
n_folds = (total_days - TRAIN_PERIOD) // STEP

print(f"Walk-Forward Configuration:")
print(f"  Training period: {TRAIN_PERIOD} days")
print(f"  Test period: {TEST_PERIOD} days")
print(f"  Step size: {STEP} days")
print(f"  Number of folds: {n_folds}")

In [ ]:
# Walk-forward validation
wf_results = []

for fold in range(n_folds):
    # Define train/test indices
    train_start = fold * STEP
    train_end = train_start + TRAIN_PERIOD
    test_start = train_end
    test_end = min(test_start + TEST_PERIOD, len(close))
    
    if test_end <= test_start:
        break
    
    # Get train/test data
    train_close = close.iloc[train_start:train_end]
    train_live = liveliness_series.iloc[train_start:train_end]
    
    test_close = close.iloc[test_start:test_end]
    test_live = liveliness_series.iloc[test_start:test_end]
    
    # TRAIN: Find best threshold on training data
    best_sharpe = -np.inf
    best_thresh = None
    
    for thresh in thresholds:
        in_sig = train_live < thresh
        entries_train = in_sig & ~in_sig.shift(1).fillna(False)
        exits_train = ~in_sig & in_sig.shift(1).fillna(False)
        
        if entries_train.sum() < 2:
            continue
            
        pf_train = vbt.Portfolio.from_signals(
            close=train_close,
            entries=entries_train,
            exits=exits_train,
            init_cash=100_000,
            fees=0.001,
            freq='D'
        )
        
        sharpe = pf_train.sharpe_ratio()
        if sharpe > best_sharpe:
            best_sharpe = sharpe
            best_thresh = thresh
    
    if best_thresh is None:
        continue
    
    # TEST: Apply best threshold to test data
    in_sig_test = test_live < best_thresh
    entries_test = in_sig_test & ~in_sig_test.shift(1).fillna(False)
    exits_test = ~in_sig_test & in_sig_test.shift(1).fillna(False)
    
    pf_test = vbt.Portfolio.from_signals(
        close=test_close,
        entries=entries_test,
        exits=exits_test,
        init_cash=100_000,
        fees=0.001,
        freq='D'
    )
    
    # Buy & hold benchmark for test period
    pf_hold_test = vbt.Portfolio.from_holding(test_close, init_cash=100_000, freq='D')
    
    wf_results.append({
        'fold': fold,
        'train_start': close.index[train_start].date(),
        'train_end': close.index[train_end-1].date(),
        'test_start': close.index[test_start].date(),
        'test_end': close.index[test_end-1].date(),
        'best_threshold': best_thresh,
        'train_sharpe': best_sharpe,
        'test_sharpe': pf_test.sharpe_ratio(),
        'test_return': pf_test.total_return(),
        'hold_return': pf_hold_test.total_return(),
        'excess_return': pf_test.total_return() - pf_hold_test.total_return(),
        'test_max_dd': pf_test.max_drawdown(),
        'n_trades': pf_test.trades.count()
    })
    
    print(f"Fold {fold}: Train {close.index[train_start].date()} - {close.index[train_end-1].date()} | "
          f"Test {close.index[test_start].date()} - {close.index[test_end-1].date()} | "
          f"Threshold: {best_thresh:.4f} | Test Sharpe: {pf_test.sharpe_ratio():.2f}")

wf_df = pd.DataFrame(wf_results)
print(f"\nCompleted {len(wf_df)} walk-forward folds")

In [ ]:
# Walk-forward results summary
print("\n" + "="*60)
print("WALK-FORWARD VALIDATION RESULTS")
print("="*60)

print(f"\n{'Metric':<30} {'Value':>15}")
print("-"*50)
print(f"{'Total Folds':<30} {len(wf_df):>15}")
print(f"{'Avg Test Sharpe':<30} {wf_df['test_sharpe'].mean():>15.2f}")
print(f"{'Avg Test Return':<30} {wf_df['test_return'].mean()*100:>14.1f}%")
print(f"{'Avg Hold Return':<30} {wf_df['hold_return'].mean()*100:>14.1f}%")
print(f"{'Avg Excess Return':<30} {wf_df['excess_return'].mean()*100:>14.1f}%")
print(f"{'Win Rate (vs Hold)':<30} {(wf_df["excess_return"] > 0).mean()*100:>14.1f}%")
print(f"{'Avg Max Drawdown':<30} {wf_df['test_max_dd'].mean()*100:>14.1f}%")

# Threshold stability
print(f"\n{'Threshold Range':<30} {wf_df['best_threshold'].min():.4f} - {wf_df['best_threshold'].max():.4f}")
print(f"{'Threshold Std Dev':<30} {wf_df['best_threshold'].std():>15.4f}")

In [ ]:
# Visualize walk-forward results
fig = vbt.make_subplots(rows=2, cols=2,
                        subplot_titles=['Test Sharpe by Fold', 
                                       'Test vs Hold Returns',
                                       'Selected Threshold by Fold',
                                       'Cumulative Excess Return'])

# Test Sharpe
colors = ['green' if s > 0 else 'red' for s in wf_df['test_sharpe']]
fig.add_bar(x=wf_df['fold'], y=wf_df['test_sharpe'], 
            marker_color=colors, name='Test Sharpe', row=1, col=1)
fig.add_hline(y=0, line_dash='dash', row=1, col=1)

# Test vs Hold Returns
fig.add_bar(x=wf_df['fold'], y=wf_df['test_return']*100, 
            name='Strategy', row=1, col=2)
fig.add_scatter(x=wf_df['fold'], y=wf_df['hold_return']*100, 
                mode='markers', marker=dict(size=10, symbol='diamond'),
                name='Buy & Hold', row=1, col=2)

# Threshold stability
fig.add_scatter(x=wf_df['fold'], y=wf_df['best_threshold'], 
                mode='lines+markers', name='Threshold', row=2, col=1)

# Cumulative excess return
cum_excess = (1 + wf_df['excess_return']).cumprod() - 1
fig.add_scatter(x=wf_df['fold'], y=cum_excess*100, 
                mode='lines+markers', name='Cum Excess', row=2, col=2)
fig.add_hline(y=0, line_dash='dash', row=2, col=2)

fig.update_layout(height=700, showlegend=True)
fig.show()

In [ ]:
# Full walk-forward results table
wf_df.round(4)

---
## 6. Performance Analysis

In [ ]:
# Use the overall best threshold for detailed analysis
FINAL_THRESHOLD = best['threshold']

# Generate final signals
final_in_signal = liveliness_series < FINAL_THRESHOLD
final_entries = final_in_signal & ~final_in_signal.shift(1).fillna(False)
final_exits = ~final_in_signal & final_in_signal.shift(1).fillna(False)

# Run final backtest
pf_final = vbt.Portfolio.from_signals(
    close=close,
    entries=final_entries,
    exits=final_exits,
    init_cash=100_000,
    fees=0.001,
    slippage=0.001,
    freq='D'
)

print(f"Final Strategy: Buy when liveliness < {FINAL_THRESHOLD:.4f}")
print("\n" + "="*60)
print(pf_final.stats())

In [ ]:
# Detailed trade analysis
print("\n" + "="*60)
print("TRADE ANALYSIS")
print("="*60)
print(pf_final.trades.stats())

In [ ]:
# Individual trades
trades_df = pf_final.trades.records_readable
print(f"\nAll {len(trades_df)} trades:")
trades_df[['Entry Timestamp', 'Exit Timestamp', 'PnL', 'Return', 'Duration']].head(20)

In [ ]:
# Drawdown analysis
print("\n" + "="*60)
print("DRAWDOWN ANALYSIS")
print("="*60)
print(pf_final.drawdowns.stats())

In [ ]:
# Plot drawdowns
pf_final.drawdowns.plot().show()

In [ ]:
# Monthly returns heatmap
pf_final.returns().vbt.returns.qs.plot_monthly_returns().show()

---
## 7. Final Summary

In [ ]:
print("\n" + "="*70)
print("LIVELINESS SIGNAL - FINAL SUMMARY")
print("="*70)

print(f"\n📊 SIGNAL SPECIFICATION")
print(f"   Metric: Liveliness")
print(f"   Direction: BUY when liveliness < {FINAL_THRESHOLD:.4f}")
print(f"   Economic rationale: Low liveliness = dormant coins = HODLers accumulating")

print(f"\n📈 IN-SAMPLE PERFORMANCE (Full Period)")
print(f"   Total Return: {pf_final.total_return()*100:.1f}%")
print(f"   Sharpe Ratio: {pf_final.sharpe_ratio():.2f}")
print(f"   Max Drawdown: {pf_final.max_drawdown()*100:.1f}%")
print(f"   Win Rate: {pf_final.trades.win_rate()*100:.1f}%")
print(f"   Total Trades: {pf_final.trades.count()}")

print(f"\n🔍 WALK-FORWARD VALIDATION (Out-of-Sample)")
print(f"   Folds tested: {len(wf_df)}")
print(f"   Avg Test Sharpe: {wf_df['test_sharpe'].mean():.2f}")
print(f"   Avg Test Return: {wf_df['test_return'].mean()*100:.1f}%")
print(f"   Beat Buy&Hold: {(wf_df['excess_return'] > 0).mean()*100:.0f}% of periods")

print(f"\n⚠️  ROBUSTNESS CHECKS")
print(f"   Sharpe curve smoothness: {smoothness:.2f} ({'✓ Smooth' if smoothness < 0.5 else '✗ Spiky'})")
print(f"   Threshold stability: {wf_df['best_threshold'].std():.4f} std dev")
print(f"   Regression p-value: < 0.001 (highly significant)")
print(f"   Cycle consistency: 4/5 bull markets (80%)")

print("\n" + "="*70)

In [ ]:
# Save results
results_summary = {
    'signal': 'liveliness',
    'direction': 'below',
    'threshold': FINAL_THRESHOLD,
    'in_sample_sharpe': pf_final.sharpe_ratio(),
    'in_sample_return': pf_final.total_return(),
    'oos_sharpe': wf_df['test_sharpe'].mean(),
    'oos_return': wf_df['test_return'].mean(),
    'beat_hold_pct': (wf_df['excess_return'] > 0).mean(),
    'smoothness': smoothness,
    'n_trades': pf_final.trades.count()
}

pd.Series(results_summary).to_json('../data/liveliness_backtest_results.json')
print("Results saved to ../data/liveliness_backtest_results.json")